In [3]:
import torch

checkpoint = torch.load('gnn_outputs.pt', weights_only=False)

graph_data = checkpoint['graph_data']
pred = checkpoint['pred']
probs = checkpoint['probs']
test_idx = checkpoint['test_idx']


fraud_probs = probs[:, 1]

In [11]:
import joblib
scaler = joblib.load('scaler.pkl')

In [4]:
print("graph_data.x:", graph_data.x.shape)
print("edge_index:", graph_data.edge_index.shape)
print("pred:", pred.shape)
print("probs:", probs.shape)
print("fraud_probs:", fraud_probs.shape)

graph_data.x: torch.Size([223000, 6])
edge_index: torch.Size([2, 600000])
pred: torch.Size([223000])
probs: torch.Size([223000, 2])
fraud_probs: torch.Size([223000])


In [6]:
print("x dtype:", graph_data.x.dtype)
print("pred dtype:", pred.dtype)
print("fraud_probs dtype:", fraud_probs.dtype)

x dtype: torch.float32
pred dtype: torch.int32
fraud_probs dtype: torch.float32


In [7]:
print("Fraud probs min:", fraud_probs.min().item())
print("Fraud probs max:", fraud_probs.max().item())

Fraud probs min: 0.020553329959511757
Fraud probs max: 0.9624485373497009


In [8]:
print("Fraud count:", pred.sum().item())
print("Total nodes:", len(pred))

Fraud count: 83554
Total nodes: 223000


In [29]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [44]:
def generate_llm_output(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
    repetition_penalty=2.0   
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [51]:
def build_reason(label, risk, freq, amount, fraud_neighbors, total_neighbors):
    
    reasons = []

    # Fraud
    if label == "FRAUD":
        
        if freq > 0:
            reasons.append("higher transaction frequency")
        
        if amount > 0:
            reasons.append("unusual transaction amounts")

        if total_neighbors > 0:
            ratio = fraud_neighbors / total_neighbors
            if ratio > 0.5:
                reasons.append("strong connections to fraudulent accounts")
            elif ratio > 0:
                reasons.append("links to suspicious accounts")

        if risk in ["HIGH RISK", "MEDIUM RISK"]:
            reasons.append("elevated risk profile")

    # No Fraud
    else:
        
        if freq < 0:
            reasons.append("low transaction activity")
        
        if amount < 0:
            reasons.append("normal transaction amounts")

        if total_neighbors > 0:
            ratio = fraud_neighbors / total_neighbors
            if ratio == 0:
                reasons.append("no connections to fraudulent accounts")
            elif ratio < 0.5:
                reasons.append("limited exposure to suspicious accounts")

        if risk == "LOW RISK":
            reasons.append("low risk profile")

    return reasons

In [62]:
def generate_explanation(i):
    
    # Feature Extraction
    node_features = graph_data.x[i].cpu().numpy()

    scaled_part = node_features[:5].reshape(1, -1)
    original_part = scaler.inverse_transform(scaled_part)[0]

    amount = round(original_part[0], 2)
    freq = round(original_part[2], 2)

    edge_index = graph_data.edge_index

    neighbors_out = edge_index[1][edge_index[0] == i]
    neighbors_in = edge_index[0][edge_index[1] == i]
    neighbors = torch.cat([neighbors_out, neighbors_in]).unique()

    fraud_neighbors = int(pred[neighbors].sum().item()) if len(neighbors) > 0 else 0
    total_neighbors = len(neighbors)

    confidence = round(fraud_probs[i].item(), 3)

    # Risk Level
    if confidence > 0.75:
        risk = "HIGH RISK"
    elif confidence > 0.5:
        risk = "MEDIUM RISK"
    else:
        risk = "LOW RISK"

    label = "FRAUD" if pred[i] == 1 else "NOT FRAUD"

    # Building reason( base logic)
    reasons = build_reason(label, risk, freq, amount, fraud_neighbors, total_neighbors)

    reason_text = ", ".join(reasons)

    # Prompt
    prompt = f"""
    Generate ONE short sentence explaining fraud detection.

    Use ONLY these keywords:
    {reason_text}

    Do NOT:
    - add new information
    - explain general concepts
    - give definitions
    - mention investigation or external facts

    Just combine the keywords into a sentence.

    Answer:
    """

    # GenAI 
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
    **inputs,
    max_new_tokens=25,     # shorter = safer
    do_sample=False,
    repetition_penalty=2.5 # stronger control
    )

    explanation = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    
    bad_phrases = ["investigation", "process", "crime", "company", "definition"]

    if any(word in explanation.lower() for word in bad_phrases) or len(explanation) < 15:
        explanation = "The account shows " + reason_text + ", indicating " + \
                  ("fraudulent behavior." if label == "FRAUD" else "normal behavior.")

    return f"""
    ==============================
    Account ID: {i}
    Prediction: {label}

    AI Explanation:
    {explanation}
    """

In [64]:
import random

sample_test_idx = random.sample(test_idx, 10)

for i in sample_test_idx:
    print(generate_explanation(i))



    Account ID: 175915
    Prediction: NOT FRAUD

    AI Explanation:
    low transaction activity, normal transaction amounts, low risk profile
    

    Account ID: 145348
    Prediction: NOT FRAUD

    AI Explanation:
    The account shows normal transaction amounts, low risk profile, indicating normal behavior.
    

    Account ID: 44143
    Prediction: NOT FRAUD

    AI Explanation:
    The account shows normal transaction amounts, low risk profile, indicating normal behavior.
    

    Account ID: 92869
    Prediction: FRAUD

    AI Explanation:
    The account shows strong connections to fraudulent accounts, elevated risk profile, indicating fraudulent behavior.
    

    Account ID: 136977
    Prediction: NOT FRAUD

    AI Explanation:
    The account shows normal transaction amounts, low risk profile, indicating normal behavior.
    

    Account ID: 55637
    Prediction: FRAUD

    AI Explanation:
    fraud detection is a method of tracking suspicious transactions.
    

  

In [16]:
print(type(scaler))

<class 'sklearn.preprocessing._data.StandardScaler'>
